# TRIAGE-EG Stage 1B — CLIP Encoder Compatibility Validation

Reuses the completed Stage 0 contracts and Stage 1A index. It never rebuilds either stage, downloads a model, scans the full dataset, or treats compatibility/smoke results as retrieval-quality evidence. Dataset and model inputs are read-only.

Default Kaggle inputs are wired to `irthn1311/triage-eg-stage0-audit-bundle` and `irthn1311/triage-eg-stage1-baseline`. Environment variables can override either mount.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"
DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE0_ROOT = Path(os.environ.get("AIC_STAGE0_ROOT", "/kaggle/working/triage_eg_stage0_audit"))
STAGE0_BUNDLE = os.environ.get(
    "AIC_STAGE0_BUNDLE",
    "/kaggle/input/datasets/irthn1311/triage-eg-stage0-audit-bundle",
)
STAGE1_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1-baseline",
    )
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1B_OUTPUT_ROOT", "/kaggle/working/triage_eg_stage1b_encoder_compatibility"
    )
)
CANDIDATE_CONFIG_VALUE = os.environ.get(
    "AIC_ENCODER_CANDIDATE_CONFIG", "configs/retrieval/stage1b_encoder_candidates.yaml"
)
CANDIDATE_IDS = tuple(
    value.strip()
    for value in os.environ.get("AIC_ENCODER_CANDIDATE_IDS", "").split(",")
    if value.strip()
)
MODEL_ROOT = Path(os.environ.get("AIC_ENCODER_MODEL_ROOT", "/kaggle/input"))
SAMPLE_SIZE = int(os.environ.get("AIC_STAGE1B_SAMPLE_SIZE", "50"))
SEED = int(os.environ.get("AIC_STAGE1B_SEED", "2026"))
REUSE_RESULTS = os.environ.get("AIC_STAGE1B_REUSE_RESULTS", "0") == "1"
RUN_TEXT_SMOKE = os.environ.get("AIC_STAGE1B_RUN_TEXT_SMOKE", "1") == "1"
print(
    {
        "ref": REPO_REF,
        "data": str(DATA_ROOT),
        "stage0_bundle": STAGE0_BUNDLE,
        "stage1": str(STAGE1_ROOT),
        "output": str(OUTPUT_ROOT),
        "model_root": str(MODEL_ROOT),
    }
)

In [ ]:
def git_result(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)


def git(*args, cwd=None):
    result = git_result(*args, cwd=cwd)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()


if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
target = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f"origin/{REPO_REF}"):
        probe = git_result("rev-parse", "--verify", f"{candidate}^{{commit}}", cwd=REPO_DIR)
        if probe.returncode == 0:
            target = probe.stdout.strip()
            break
if target is None:
    git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
    target = "FETCH_HEAD"
git("checkout", "--detach", target, cwd=REPO_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["AIC_RESOLVED_GIT_COMMIT"] = COMMIT
sys.path.insert(0, str(REPO_DIR / "src"))
print("resolved commit:", COMMIT)

In [ ]:
from triage_eg.retrieval.stage1.stage0_loader import resolve_stage0_root
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root

if not DATA_ROOT.is_dir():
    raise RuntimeError(f"Missing dataset root: {DATA_ROOT}")
STAGE0_ROOT = resolve_stage0_root(
    STAGE0_ROOT,
    bundle_path=STAGE0_BUNDLE or None,
    search_root=Path("/kaggle/input"),
    excluded_roots=(DATA_ROOT,),
)
STAGE1_ROOT = resolve_stage1_root(
    STAGE1_ROOT,
    search_root=Path("/kaggle/input"),
    excluded_roots=(DATA_ROOT, STAGE0_ROOT),
)
print("resolved Stage 0 root:", STAGE0_ROOT)
print("resolved Stage 1A root:", STAGE1_ROOT)
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
probe_path = OUTPUT_ROOT.parent / ".stage1b_write_probe"
probe_path.write_text("ok", encoding="utf-8")
probe_path.unlink()
CANDIDATE_CONFIG = Path(CANDIDATE_CONFIG_VALUE)
if not CANDIDATE_CONFIG.is_absolute():
    CANDIDATE_CONFIG = REPO_DIR / CANDIDATE_CONFIG
for dependency in ("clip", "open_clip", "torch", "PIL"):
    print(dependency, "available=", importlib.util.find_spec(dependency) is not None)
print("No pip install and no model download will be performed.")

In [ ]:
stage1_summary = json.loads((STAGE1_ROOT / "stage1_summary.json").read_text())
stage1_manifest = json.loads((STAGE1_ROOT / "index/index_manifest.json").read_text())
if stage1_summary.get("index_status") != "COMPLETE":
    raise RuntimeError("Stage 1A index status is not COMPLETE")
if stage1_summary.get("next_stage_readiness", {}).get("corpus_index") not in {
    "READY",
    "READY_WITH_TIE_WARNINGS",
}:
    raise RuntimeError("Stage 1A corpus index is not ready")
print(
    {
        "vectors": stage1_manifest["vector_count"],
        "dimension": stage1_manifest["dimension"],
        "index_fingerprint": stage1_summary["index_fingerprint"],
        "self_retrieval": stage1_summary["self_retrieval_status"],
        "encoder_blocker": stage1_summary["next_stage_readiness"]["text_retrieval"],
    }
)

In [ ]:
from triage_eg.retrieval.stage1b.evidence import discover_encoder_evidence

evidence, evidence_summary = discover_encoder_evidence(REPO_DIR, DATA_ROOT)
print(evidence_summary)
for item in evidence[:20]:
    print(item)

In [ ]:
from triage_eg.retrieval.stage1b.assets import preflight_candidate
from triage_eg.retrieval.stage1b.registry import load_candidate_registry

candidate_contracts, gate, _, config_fingerprint = load_candidate_registry(
    CANDIDATE_CONFIG, CANDIDATE_IDS
)
for candidate in candidate_contracts:
    provenance, candidate_issues = (
        preflight_candidate(candidate, REPO_DIR, DATA_ROOT)
        if candidate.enabled
        else ({"status": "DISABLED"}, [])
    )
    print(
        candidate.candidate_id,
        candidate.enabled,
        candidate.implementation,
        provenance,
        candidate_issues,
    )
print("threshold config fingerprint:", config_fingerprint)

In [ ]:
from triage_eg.retrieval.stage1b.sampling import select_probe_samples

probe_samples, sample_issues = select_probe_samples(STAGE1_ROOT, DATA_ROOT, SAMPLE_SIZE, SEED)
for item in probe_samples[:10]:
    print(item)
print("sample issues:", sample_issues)

In [ ]:
from triage_eg.retrieval.stage1b import Stage1BConfig, run_stage1b

result = run_stage1b(
    Stage1BConfig(
        repo_root=REPO_DIR,
        dataset_root=DATA_ROOT,
        stage0_root=STAGE0_ROOT,
        stage1_root=STAGE1_ROOT,
        output_root=OUTPUT_ROOT,
        candidate_config=CANDIDATE_CONFIG,
        smoke_queries=REPO_DIR / "configs/retrieval/stage1b_smoke_queries.jsonl",
        sample_size=SAMPLE_SIZE,
        seed=SEED,
        candidate_ids=CANDIDATE_IDS,
        overwrite=not REUSE_RESULTS,
        reuse_results=REUSE_RESULTS,
        strict_root=True,
        run_text_smoke=RUN_TEXT_SMOKE,
        build_git_commit=COMMIT,
    )
)
print("Stage 1B reused:", result.reused)

In [ ]:
candidate_summaries = [
    json.loads(line)
    for line in (OUTPUT_ROOT / "probe/candidate_summaries.jsonl").read_text().splitlines()
    if line.strip()
]
for item in candidate_summaries:
    print(
        {
            "candidate": item["candidate_id"],
            "completed": item["samples_completed"],
            "cosine": item["cosine"],
            "alignment": item["retrieval_alignment"],
            "decision": item["decision"],
            "reasons": item["decision_reasons"],
        }
    )

In [ ]:
selected = json.loads((OUTPUT_ROOT / "encoder/selected_encoder_contract.json").read_text())
print(json.dumps(selected, indent=2, ensure_ascii=False))

In [ ]:
smoke_results = [
    json.loads(line)
    for line in (OUTPUT_ROOT / "smoke/smoke_results.jsonl").read_text().splitlines()
    if line.strip()
]
if selected.get("compatibility_status") != "VERIFIED":
    print("Text smoke BLOCKED:", selected.get("reason"))
else:
    for item in smoke_results:
        print(item)

In [ ]:
for item in smoke_results:
    frames = OUTPUT_ROOT / item["result_artifacts"]["ranked_frames"]
    top_results = [json.loads(line) for line in frames.read_text().splitlines()[:20]]
    print(item["query_id"], item["text"], top_results)

In [ ]:
summary = json.loads((OUTPUT_ROOT / "stage1b_summary.json").read_text())
print("encoder compatibility:", summary["readiness"]["encoder_compatibility"])
print("text retrieval readiness:", summary["readiness"]["text_retrieval"])
print("remaining blockers:", summary["issues"])
print("non-claims:", summary["non_claims"])

In [ ]:
from zipfile import ZipFile

from triage_eg.retrieval.stage1b.writers import create_stage1b_report_bundle

zip_path = Path("/kaggle/working/triage_eg_stage1b_encoder_compatibility_reports.zip")
create_stage1b_report_bundle(OUTPUT_ROOT, zip_path)
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert not any(
    name.endswith((".bin", ".pt", ".pth", ".npy", ".jpg")) or name.startswith("logs/")
    for name in members
)
assert zip_path.name not in members
print("DOWNLOAD ZIP:", zip_path, "size_bytes=", zip_path.stat().st_size)
print("members:", members)
print(
    "This notebook validates encoder compatibility. It does not rebuild the BTC index "
    "or claim semantic retrieval quality."
)